# Insurgent Primaries, 2018–2026: Money, Outside Spending, and Outcomes

The case universe is every House candidate endorsed by Justice Democrats from 2018 through 2026 (144 endorsement rows from the organization's Wikipedia tables, which carry primary result and vote share), matched to FEC data for the candidate and for their main Democratic-primary opponent — the incumbent if the JD candidate was a challenger, else the best-funded other Democrat in the district. Justice Democrats is used as the universe because it is the one insurgent-left organization with a complete, dated endorsement record across all five cycles; DSA's federal endorsements overlap heavily with it (Ocasio-Cortez, Tlaib, Bush, Bowman, Lee, Casar, Kiros, Valdez, Avila Chevalier, McKinney).

**Questions.** Do JD candidates win when outspent, and how often? Does outside spending against them predict losing? How do the money ratios in the wins (Ocasio-Cortez 2018, Bowman 2020, Bush 2020, Lee 2022, Kiros / McKinney / Avila Chevalier 2026) compare with the losses (Cisneros 2020 and 2022, Turner 2021, Bowman and Bush 2024, Allam and Abughazaleh 2026)?

**Inputs:** `data/raw/insurgent/jd_cases_fec.csv` from `src/data_collection/fec_insurgent_pull.py`. Money is receipts through the last FEC report filed before the primary (year-to-date plus the prior year-end), which is the cleanest pre-primary measure available without itemized-transaction pulls; independent expenditures are Schedule E totals for the cycle by support/oppose.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')
RAW_DIR = Path('../data/raw'); TABLES_DIR = Path('../outputs/tables')
pd.set_option('display.width', 240)
d = pd.read_csv(RAW_DIR / 'insurgent' / 'jd_cases_fec.csv')
# 'Advanced' = made a top-two general (CA/WA) or a runoff; runoff outcomes and blank results are set by hand from the Wikipedia table
OVERRIDE = {(2022, 'Jessica Cisneros'): ('Lost', 49.69), (2022, 'Kina Collins'): ('Lost', 45.72), (2022, 'Marie Newman'): ('Lost', 29.23), (2018, 'Lorie Burch'): ('Won', 75.04)}
for (cy, nm), (res, pct) in OVERRIDE.items():
    d.loc[(d.cycle == cy) & (d.candidate == nm), ['prim_result', 'prim_pct']] = [res, f'{pct}%']
d['won'] = d.prim_result.isin(['Won', 'Advanced']).astype(float)
d['pct'] = pd.to_numeric(d.prim_pct.astype(str).str.rstrip('%'), errors='coerce')
d['contested'] = d.pct < 99
d['type'] = np.select([d.jd_incumbent, d.opp_is_incumbent], ['JD incumbent', 'challenger vs incumbent'], 'open / other')
for c in ['pre_primary_receipts', 'opp_pre_primary_receipts', 'ie_support', 'ie_oppose', 'opp_ie_support', 'opp_ie_oppose', 'cycle_receipts', 'opp_cycle_receipts']:
    d[c] = pd.to_numeric(d[c], errors='coerce').fillna(0) / 1e6
d['jd_side'] = d.pre_primary_receipts + d.ie_support + d.opp_ie_oppose
d['opp_side'] = d.opp_pre_primary_receipts + d.opp_ie_support + d.ie_oppose
d['money_ratio'] = d.opp_side / d.jd_side.replace(0, np.nan)
d['outspent'] = d.opp_side > d.jd_side
d['small_share'] = pd.to_numeric(d.pre_primary_unitemized, errors='coerce') / (pd.to_numeric(d.pre_primary_receipts, errors='coerce') * 1e6)
print(len(d), 'cases;', d.contested.sum(), 'contested')
print(d.groupby(['cycle', 'type']).size().unstack(fill_value=0))

133 cases; 111 contested
type   JD incumbent  challenger vs incumbent  open / other
cycle                                                     
2018              3                        9            54
2020              7                        6             1
2021              0                        0             1
2022             10                        3             3
2024             12                        0             0
2026              9                        7             8


## 1. Win rates by case type, cycle, and money position

In [2]:
c = d[d.contested].copy()
print('Contested primaries only (JD candidate < 99%):')
print(c.groupby('type').apply(lambda s: pd.Series({'N': len(s), 'wins': s.won.sum(), 'win_rate': s.won.mean(), 'outspent_share': s.outspent.mean(), 'median_money_ratio': s.money_ratio.median()})).round(2))
print()
print(c.groupby('cycle').apply(lambda s: pd.Series({'N': len(s), 'wins': s.won.sum(), 'win_rate': s.won.mean(), 'outspent_share': s.outspent.mean()})).round(2))
print()
print('Win rate by whether the JD side was outspent (contested):')
print(c.groupby(['type', 'outspent']).apply(lambda s: pd.Series({'N': len(s), 'win_rate': s.won.mean(), 'mean_pct': s.pct.mean()})).round(2))
c.groupby(['type', 'outspent']).apply(lambda s: pd.Series({'N': len(s), 'win_rate': s.won.mean()})).to_csv(TABLES_DIR / 'insurgent_winrate_by_money.csv')

Contested primaries only (JD candidate < 99%):
                            N  wins  win_rate  outspent_share  median_money_ratio
type                                                                             
JD incumbent             28.0  25.0      0.89            0.11                0.01
challenger vs incumbent  24.0  11.0      0.46            0.79                2.41
open / other             59.0  21.0      0.36            0.56                1.28

          N  wins  win_rate  outspent_share
cycle                                      
2018   58.0  18.0      0.31            0.66
2020   12.0   9.0      0.75            0.42
2021    1.0   0.0      0.00            0.00
2022   11.0   8.0      0.73            0.18
2024    7.0   5.0      0.71            0.29
2026   22.0  17.0      0.77            0.36

Win rate by whether the JD side was outspent (contested):
                                     N  win_rate  mean_pct
type                    outspent                          
JD incumbent 

## 2. The challenger-vs-incumbent cases, one by one

In [3]:
ch = c[c.type == 'challenger vs incumbent'].sort_values(['cycle', 'pct'], ascending=[True, False])
cols = ['cycle', 'candidate', 'state', 'district', 'opp_name', 'prim_result', 'pct', 'pre_primary_receipts', 'opp_pre_primary_receipts', 'ie_support', 'ie_oppose', 'opp_ie_support', 'opp_ie_oppose', 'money_ratio', 'small_share', 'ie_top_opposers']
print(ch[cols].round(2).to_string(index=False))
ch[cols].to_csv(TABLES_DIR / 'insurgent_challengers_vs_incumbents.csv', index=False)

 cycle                 candidate state  district                         opp_name prim_result   pct  pre_primary_receipts  opp_pre_primary_receipts  ie_support  ie_oppose  opp_ie_support  opp_ie_oppose  money_ratio  small_share                                                                                     ie_top_opposers
  2018           Ayanna Pressley    MA         7               CAPUANO, MICHAEL E         Won 58.60                  0.90                      2.08        0.10       0.00            0.28           0.00         2.36         0.22                                                                                                 NaN
  2018  Alexandria Ocasio-Cortez    NY        14                  CROWLEY, JOSEPH         Won 56.74                  0.37                      5.54        0.00       0.63            0.00           0.01        15.99         0.69                                                                                      FUTURE45 0.63M
  2018          

## 3. Outside spending against insurgents: who, how much, and did it work?

In [4]:
d['ie_against_any'] = d.ie_oppose > 0.25
print('Cases with > $250k in independent expenditures against the JD candidate:')
big = d[d.ie_against_any].sort_values('ie_oppose', ascending=False)
print(big[['cycle', 'candidate', 'state', 'district', 'type', 'prim_result', 'pct', 'ie_oppose', 'ie_support', 'jd_side', 'opp_side', 'ie_top_opposers']].round(2).to_string(index=False))
print('\nWin rate: with > $250k against %.2f (N=%d); without %.2f (N=%d) [contested only]' % (
    c[c.ie_oppose > 0.25].won.mean(), (c.ie_oppose > 0.25).sum(), c[c.ie_oppose <= 0.25].won.mean(), (c.ie_oppose <= 0.25).sum()))
# who spends against: aggregate top opposers text
import re
from collections import Counter
cnt = Counter()
for s in d.ie_top_opposers.dropna():
    for part in s.split(';'):
        m = re.match(r'\s*(.+?)\s+([\d.]+)M', part)
        if m: cnt[m.group(1).strip()] += float(m.group(2))
print('\nLargest opposing committees across all cases ($M, top-3 per case summed):')
for k, v in cnt.most_common(12): print(f'  {k[:50]:50s} {v:6.2f}')

Cases with > $250k in independent expenditures against the JD candidate:
 cycle                 candidate state  district                    type prim_result   pct  ie_oppose  ie_support  jd_side  opp_side                                                                                                          ie_top_opposers
  2024             Jamaal Bowman    NY        16            JD incumbent        Lost 41.41      12.04        1.89     8.82     24.80                                                  UNITED DEMOCRACY PROJECT ('UDP') 9.87M; FAIRSHAKE 2.08M; DMFI PAC 0.10M
  2024                 Cori Bush    MO         1            JD incumbent        Lost 45.64       7.34        2.24     7.31     17.47                                  UNITED DEMOCRACY PROJECT ('UDP') 5.24M; FAIRSHAKE 1.41M; MAINSTREAM DEMOCRATS PAC 0.60M
  2020              Kara Eastman    NE         2            open / other         Won 62.15       5.45        0.89     2.21      6.10                               NR

## 4. Models: money ratio, outside spending, incumbency, and the vote share

In [5]:
m = c.dropna(subset=['money_ratio']).copy()
m['log_ratio'] = np.log(m.money_ratio.clip(0.05, 50))
m['log_ie_oppose'] = np.log1p(m.ie_oppose)
m['vs_inc'] = (m.type == 'challenger vs incumbent').astype(float); m['jd_inc'] = (m.type == 'JD incumbent').astype(float)
m['post2020'] = (m.cycle >= 2022).astype(float)
X = sm.add_constant(m[['log_ratio', 'log_ie_oppose', 'vs_inc', 'jd_inc', 'post2020']])
r = sm.OLS(m.pct, X).fit(cov_type='HC1'); print('Primary vote share (%%) on money and type, contested cases (N=%d):' % len(m)); print(pd.DataFrame({'coef': r.params, 'se': r.bse, 'p': r.pvalues}).round(2))
r2 = sm.Logit(m.won, X).fit(disp=0, cov_type='HC1'); print('\nWin (logit):'); print(pd.DataFrame({'coef': r2.params, 'se': r2.bse, 'p': r2.pvalues}).round(2))
pd.DataFrame({'ols_coef': r.params, 'ols_se': r.bse, 'logit_coef': r2.params, 'logit_se': r2.bse}).to_csv(TABLES_DIR / 'insurgent_models.csv')

Primary vote share (%) on money and type, contested cases (N=111):
                coef    se     p
const          30.38  2.59  0.00
log_ratio      -6.85  1.30  0.00
log_ie_oppose   4.85  3.46  0.16
vs_inc         15.17  3.15  0.00
jd_inc         18.45  4.96  0.00
post2020        2.69  3.20  0.40

Win (logit):
               coef    se     p
const         -0.64  0.31  0.04
log_ratio     -0.48  0.19  0.01
log_ie_oppose -0.44  0.61  0.47
vs_inc         0.75  0.58  0.20
jd_inc         1.54  0.76  0.04
post2020       0.77  0.55  0.16


## 5. Small-dollar share

In [6]:
print('Small-dollar (unitemized) share of pre-primary receipts, JD candidates vs opponents:')
d['opp_small_share'] = pd.to_numeric(d.opp_pre_primary_unitemized, errors='coerce') / (pd.to_numeric(d.opp_pre_primary_receipts, errors='coerce') * 1e6)
print(d.groupby('type')[['small_share', 'opp_small_share']].median().round(2))
print(d.groupby('cycle')[['small_share', 'opp_small_share']].median().round(2))
print('\nWinners vs losers among contested JD candidates (median):'); print(c.groupby('won')[['small_share', 'pre_primary_receipts', 'money_ratio', 'ie_oppose']].median().round(2))

Small-dollar (unitemized) share of pre-primary receipts, JD candidates vs opponents:
                         small_share  opp_small_share
type                                                 
JD incumbent                    0.28             0.05
challenger vs incumbent         0.39             0.02
open / other                    0.35             0.12
       small_share  opp_small_share
cycle                              
2018          0.35             0.12
2020          0.41             0.01
2021          0.56             0.04
2022          0.32             0.05
2024          0.24             0.07
2026          0.32             0.03

Winners vs losers among contested JD candidates (median):
     small_share  pre_primary_receipts  money_ratio  ie_oppose
won                                                           
0.0         0.35                  0.50         1.60        0.0
1.0         0.34                  2.09         0.35        0.0


## Limitations

- **Universe = Justice Democrats endorsements.** Insurgent-left candidates JD did not endorse (e.g., Brad Lander 2026, Shahid Buttar 2020) are excluded; JD's 2018 slate included many open-seat and general-election-only candidates, which is why the analysis conditions on contested primaries and separates the challenger-vs-incumbent cases.
- **Opponent identification is mechanical:** the Democratic incumbent, else the best-funded other Democrat by cycle receipts. In multi-candidate open seats the "opponent" is one of several.
- **Pre-primary receipts are year-to-date at the last report before the primary plus the prior year-end;** where the FEC has no 12P report the prior quarterly is used. Independent expenditures are cycle totals, which for incumbents include general-election spending; for primary-only losers they are effectively primary totals.
- **2026 outcomes are as recorded on Wikipedia at pull time**; 2026 FEC totals are partial (cycle still open).
- **Small N.** Roughly 25–30 challenger-vs-incumbent cases across five cycles; the models are descriptive.